In [1]:
# Importando as bibliotecas
import requests
import pandas as pd
import numpy as np
import re
import os


# * Listando os parâmetros que serão requisitados da API
parametros = {
    '@trimestre': "'20201'",
    '$top': 3026,
    '$format': 'json',
    '$select': 'trimestre,nomeBandeira,nomeFuncao,produto,modalidade,qtdCartoesEmitidos,qtdCartoesAtivos,qtdTransacoesNacionais,valorTransacoesNacionais,qtdTransacoesInternacionais,valorTransacoesInternacionais'
}

site = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/Quantidadeetransacoesdecartoes(trimestre=@trimestre)'


# * Requisição + Tratamento de Erro 
try:
    response = requests.get(url=site, params=parametros)
    response.raise_for_status()
    data = response.json()
    
    # * Salvando os dados em um DF
    dados_brutos = data['value']
    df = pd.DataFrame(dados_brutos)
    
    # * Tratamento de Dados
    df_copia = df.copy()
    
    
    # Função que insere sublinhado antes de maiúsculas e converte para minúsculas
    def camel_to_snake(name):
        
        # Adiciona '_' antes de maiúsculas e remove espaços extras
        s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
        return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()

    # Aplicando a conversão em todas as colunas
    df_copia.columns = [camel_to_snake(col) for col in df.columns]
    
    
    # / Tratamento de Dados
        # / Renomeando colunas
    df_copia = df_copia.rename(columns={
        'nome_bandeira': 'bandeira',
        'nome_funcao': 'funcao',
        'produto': 'categoria'
        })


        # / Separando o trimestre e o ano em colunas diferentes 
    df_copia['ano'] = df_copia['trimestre'].astype(str).str[-1]
    df_copia['trimestre'] = df_copia['trimestre'].astype(str).str[:4]
    
        # / Alterando o tipo do ano -> int
    df_copia['trimestre'] = df_copia['trimestre'].astype(int)
    df_copia['ano'] = df_copia['ano'].astype(int)
    
        # / Renomeando as colunas
    df_copia = df_copia.rename(columns={'trimestre': 'ano', 'ano': 'trimestre'})
    
        # / Reordenando as colunas
    coluna_trimestre = df_copia.pop('trimestre')
    df_copia.insert(1, 'trimestre', coluna_trimestre)
    
    
        # * 1. Mapeia qual é o mês e o dia final de cada número de trimestre
    fim_trimestre = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}
    
        # * 2. Junta o ano com o sufixo correspondente do trimestre
    df_copia['data_trimestre'] = (
        df_copia['ano'].astype(str) + df_copia['trimestre'].map(fim_trimestre)
    )
        # * 3. Converte para data
    df_copia['data_trimestre'] = pd.to_datetime(df_copia['data_trimestre']).dt.normalize()
    
    
        # / Reordenando coluna data_trimestre
    coluna_data_trimestre = df_copia.pop('data_trimestre')
    df_copia.pop('ano')
    df_copia.insert(0, 'data_trimestre', coluna_data_trimestre)

    # * Ordenando os dados pela data
    df_copia = df_copia.sort_values(by='data_trimestre')
    df_copia = df_copia.reset_index(drop=True)
    
    
    # * Mostrando o dataframe
    display(df_copia['bandeira'].unique())
    display(df_copia.info())
    display(df_copia)


    # * Salvando os dados em um arquivo csv
    caminho_csv = os.path.join('..', 'data', 'stg_transacoes_cartao.csv')
    
    df_copia.to_csv(caminho_csv, index=False, sep=';', encoding='utf-8-sig', float_format='%.2f')
    print(f'Arquivo salvo com sucesso em: {os.path.abspath(caminho_csv)}')
    
    
except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a API: {erro}')





array(['Elo', 'MasterCard', 'American Express', 'VISA',
       'Bandeira própria', 'Outras', 'Hipercard'], dtype=object)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3026 entries, 0 to 3025
Data columns (total 12 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   data_trimestre                   3026 non-null   datetime64[ns]
 1   trimestre                        3026 non-null   int64         
 2   bandeira                         3026 non-null   object        
 3   funcao                           3026 non-null   object        
 4   categoria                        3026 non-null   object        
 5   modalidade                       3026 non-null   object        
 6   qtd_cartoes_emitidos             3026 non-null   int64         
 7   qtd_cartoes_ativos               3026 non-null   int64         
 8   qtd_transacoes_nacionais         3026 non-null   int64         
 9   valor_transacoes_nacionais       3026 non-null   float64       
 10  qtd_transacoes_internacionais    3026 non-null   int64      

None

,data_trimestre,trimestre,bandeira,funcao,categoria,modalidade,qtd_cartoes_emitidos,qtd_cartoes_ativos,qtd_transacoes_nacionais,valor_transacoes_nacionais,qtd_transacoes_internacionais,valor_transacoes_internacionais
0,2020-03-31,1,Elo,Crédito,Empresarial,Puro,441358,193392,4342680,9.276904e+08,4465,2.454350e+06
1,2020-03-31,1,MasterCard,Crédito,Premium,Co-branded,103133,69635,2950987,6.446518e+08,130892,5.755543e+07
2,2020-03-31,1,American Express,Crédito,Platinum,Puro,466359,349194,11899109,2.847010e+09,622335,4.812225e+08
3,2020-03-31,1,MasterCard,Crédito,Básico Internacional,Puro,10832558,6690938,109172686,9.677491e+09,640775,1.273224e+08
4,2020-03-31,1,VISA,Crédito,Premium,Co-branded,212,61,827934,1.343120e+08,14496,4.587312e+06
...,...,...,...,...,...,...,...,...,...,...,...,...
3021,2026-03-31,1,MasterCard,Débito,Intermediário,Híbrido,155081,61795,777534,7.027802e+07,1,1.004300e+02
3022,2026-03-31,1,MasterCard,Débito,Básico Internacional,Híbrido,294636,85729,585411,4.710475e+07,15,1.793370e+03
3023,2026-03-31,1,VISA,Crédito,Corporativo,Puro,704689,329571,6594111,3.022052e+09,402021,4.010239e+08
3024,2026-03-31,1,MasterCard,Pré-Pago,Corporativo,Puro,1880355,98631,109150,5.902237e+06,0,0.000000e+00


Arquivo salvo com sucesso em: c:\Users\mathe\OneDrive\Documentos\Meus Projetos\Análise de Dados\Projeto end-to-end\data\stg_transacoes_cartao.csv
